In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
customers = pd.read_csv(BASE_PATH + 'customers.csv')
invoices = pd.read_csv(BASE_PATH + 'invoices.csv')
payments = pd.read_csv(BASE_PATH + 'payments.csv')

# invoices whose invoice ID appears only once
clean_invoices = invoices[~invoices['invoice_id'].duplicated(keep=False)].copy()

# payments that belong to those invoices
payments_used = payments[payments['invoice_id'].isin(clean_invoices['invoice_id'])].copy()

print("sales orders header:", so_h.shape)
print("customers:", customers.shape)
print("invoices:", invoices.shape)
print("payments:", payments.shape)
print("clean invoices:", clean_invoices.shape)
print("payments used:", payments_used.shape)
print("delivered orders:", (so_h['order_status'] == 'Delivered').sum())
print("customers with a clean invoice:", clean_invoices['customer_id'].nunique())

Mounted at /content/drive
sales orders header: (20000, 11)
customers: (500, 15)
invoices: (18033, 10)
payments: (19257, 5)
clean invoices: (17640, 10)
payments used: (18840, 5)
delivered orders: 18033
customers with a clean invoice: 500


In [2]:
orders = so_h[so_h['order_status'] == 'Delivered'].copy()
orders['order_date'] = pd.to_datetime(orders['order_date'])

snapshot = pd.Timestamp('2025-01-01')

silence = orders.groupby('customer_id').agg(
    orders=('so_id', 'count'),
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max')
).reset_index()

silence['recency_days'] = (snapshot - silence['last_order']).dt.days
silence['normal_gap'] = ((silence['last_order'] - silence['first_order']).dt.days / (silence['orders'] - 1)).round(2)
silence['silence_ratio'] = (silence['recency_days'] / silence['normal_gap']).round(2)

print("customers:", len(silence))
print("customers with silence ratio above 2:", (silence['silence_ratio'] > 2).sum())
print("customers with last order before 1 Jul 2024:", (silence['last_order'] < pd.Timestamp('2024-07-01')).sum())
print("customers in both groups:", ((silence['silence_ratio'] > 2) & (silence['last_order'] < pd.Timestamp('2024-07-01'))).sum())
print(silence[silence['customer_id'].isin(['C0001', 'C0002', 'C0003'])].to_string(index=False))

customers: 500
customers with silence ratio above 2: 66
customers with last order before 1 Jul 2024: 23
customers in both groups: 23
customer_id  orders first_order last_order  recency_days  normal_gap  silence_ratio
      C0001      34  2019-01-26 2024-11-15            47       64.24           0.73
      C0002      51  2019-01-24 2024-09-07           116       41.06           2.83
      C0003      38  2019-04-05 2024-10-30            63       55.00           1.15


In [3]:
silence['normal_gap_exact'] = (silence['last_order'] - silence['first_order']).dt.days / (silence['orders'] - 1)
silence['ratio_exact'] = silence['recency_days'] / silence['normal_gap_exact']

print("above 2 with exact numbers:", (silence['ratio_exact'] > 2).sum())
print("above 2 with rounded numbers:", (silence['silence_ratio'] > 2).sum())

near = silence[(silence['ratio_exact'] > 1.99) & (silence['ratio_exact'] < 2.01)].copy()
near['ratio_exact'] = near['ratio_exact'].round(5)
print(near[['customer_id', 'recency_days', 'normal_gap', 'silence_ratio', 'ratio_exact']].to_string(index=False))

above 2 with exact numbers: 67
above 2 with rounded numbers: 66
customer_id  recency_days  normal_gap  silence_ratio  ratio_exact
      C0083            89       44.57           2.00      1.99694
      C0299           124       61.97           2.00      2.00098
      C0417           112       55.77           2.01      2.00810


In [4]:
from scipy.stats import chi2_contingency

ci = clean_invoices.copy()
ci['terms_days'] = (pd.to_datetime(ci['due_date']) - pd.to_datetime(ci['invoice_date'])).dt.days

terms = ci.groupby('customer_id')['terms_days'].agg(['nunique', 'first']).reset_index()
print("customers with more than one due period:", (terms['nunique'] > 1).sum())

silence['quiet'] = silence['ratio_exact'] > 2
mix = silence.merge(terms[['customer_id', 'first']].rename(columns={'first': 'terms_days'}), on='customer_id')

table = pd.crosstab(mix['quiet'], mix['terms_days'])
table.index = ['not quiet', 'quiet']
print(table)
print((table.div(table.sum(axis=1), axis=0) * 100).round(1))

chi2, p, dof, expected = chi2_contingency(table)
print("chi-square:", round(chi2, 3), "| degrees of freedom:", dof, "| p-value:", round(p, 4))
print("smallest expected count:", round(expected.min(), 2))

customers with more than one due period: 0
terms_days  15   30  45  60
not quiet   74  168  99  92
quiet       13   25  14  15
terms_days    15    30    45    60
not quiet   17.1  38.8  22.9  21.2
quiet       19.4  37.3  20.9  22.4
chi-square: 0.346 | degrees of freedom: 3 | p-value: 0.9511
smallest expected count: 11.66


In [5]:
from scipy.stats import mannwhitneyu

pay = payments_used.merge(clean_invoices[['invoice_id', 'due_date']], on='invoice_id', how='left')
pay['payment_date'] = pd.to_datetime(pay['payment_date'])
pay['due_date'] = pd.to_datetime(pay['due_date'])
pay['is_late'] = pay['payment_date'] > pay['due_date']

per_invoice = pay.groupby('invoice_id').agg(paid_amount=('payment_amount', 'sum'), late=('is_late', 'max')).reset_index()

inv = clean_invoices[['invoice_id', 'customer_id', 'grand_total', 'payment_status']].merge(per_invoice, on='invoice_id', how='left')
inv['has_payment'] = inv['paid_amount'].notna()
inv['paid_amount'] = inv['paid_amount'].fillna(0)
inv['late'] = inv['late'].fillna(False).astype(bool)
inv['not_fully_paid'] = inv['payment_status'] != 'Paid'

cust = inv.groupby('customer_id').agg(
    invoices=('invoice_id', 'count'),
    with_payment=('has_payment', 'sum'),
    late_invoices=('late', 'sum'),
    not_fully_paid=('not_fully_paid', 'sum'),
    billed=('grand_total', 'sum'),
    paid=('paid_amount', 'sum')
).reset_index()

cust['late_rate'] = cust['late_invoices'] / cust['with_payment'] * 100
cust['not_fully_paid_pct'] = cust['not_fully_paid'] / cust['invoices'] * 100
cust['outstanding_pct'] = (cust['billed'] - cust['paid']) / cust['billed'] * 100
cust = cust.merge(silence[['customer_id', 'quiet']], on='customer_id')

print("customers:", len(cust), "| invoices:", cust['invoices'].sum())
print("billed:", round(cust['billed'].sum(), 2), "| paid:", round(cust['paid'].sum(), 2))
print("expected: 17640 invoices, billed 28599637860.60, paid 22795333903.87")

rows = []
for col, name in [('late_rate', 'late invoice rate %'),
                  ('not_fully_paid_pct', 'not fully paid invoices %'),
                  ('outstanding_pct', 'outstanding % of billed')]:
    a = cust.loc[cust['quiet'], col].dropna()
    b = cust.loc[~cust['quiet'], col].dropna()
    p = mannwhitneyu(a, b, alternative='two-sided').pvalue
    rows.append([name, len(a), round(a.mean(), 2), round(a.median(), 2),
                 len(b), round(b.mean(), 2), round(b.median(), 2), round(p, 4)])

result = pd.DataFrame(rows, columns=['measure', 'quiet customers', 'quiet mean', 'quiet median',
                                     'other customers', 'other mean', 'other median', 'p-value'])
print(result.to_string(index=False))
print("strict line:", round(0.05 / 3, 4))

customers: 500 | invoices: 17640
billed: 28599637860.6 | paid: 22795333903.87
expected: 17640 invoices, billed 28599637860.60, paid 22795333903.87
                  measure  quiet customers  quiet mean  quiet median  other customers  other mean  other median  p-value
      late invoice rate %               67       45.53         50.00              433       44.90         48.65   0.9290
not fully paid invoices %               67       32.01         31.58              433       30.37         29.73   0.2787
  outstanding % of billed               67       21.45         19.74              433       20.23         19.61   0.3554
strict line: 0.0167


/tmp/ipykernel_5714/2547803863.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  inv['late'] = inv['late'].fillna(False).astype(bool)


In [6]:
cust['outstanding'] = cust['billed'] - cust['paid']
total_billed = cust['billed'].sum()
total_out = cust['outstanding'].sum()

gone = silence.set_index('customer_id')['last_order'] < pd.Timestamp('2024-07-01')
cust['gone_since_june'] = cust['customer_id'].map(gone)

groups = [('quiet (ratio above 2)', cust['quiet']),
          ('no order since 30 Jun 2024', cust['gone_since_june']),
          ('all other customers', ~cust['quiet'])]

rows = []
for name, mask in groups:
    sub = cust[mask]
    rows.append([name, len(sub),
                 round(sub['billed'].sum() / total_billed * 100, 2),
                 round(sub['outstanding'].sum()),
                 round(sub['outstanding'].sum() / total_out * 100, 2),
                 round(sub['outstanding'].sum() / sub['billed'].sum() * 100, 2)])

print(pd.DataFrame(rows, columns=['group', 'customers', 'share of all billed %', 'outstanding money',
                                  'share of all outstanding %', 'outstanding % of own billed']).to_string(index=False))

rows = []
for col, name in [('late_rate', 'late invoice rate %'),
                  ('not_fully_paid_pct', 'not fully paid invoices %'),
                  ('outstanding_pct', 'outstanding % of billed')]:
    a = cust.loc[cust['gone_since_june'], col].dropna()
    b = cust.loc[~cust['gone_since_june'], col].dropna()
    p = mannwhitneyu(a, b, alternative='two-sided').pvalue
    rows.append([name, len(a), round(a.mean(), 2), round(a.median(), 2),
                 len(b), round(b.mean(), 2), round(b.median(), 2), round(p, 4)])

print(pd.DataFrame(rows, columns=['measure', 'no order since June', 'their mean', 'their median',
                                  'other customers', 'other mean', 'other median', 'p-value']).to_string(index=False))
print("strict line:", round(0.05 / 3, 4))

                     group  customers  share of all billed %  outstanding money  share of all outstanding %  outstanding % of own billed
     quiet (ratio above 2)         67                  13.12          804728429                       13.86                        21.45
no order since 30 Jun 2024         23                   4.13          266055010                        4.58                        22.53
       all other customers        433                  86.88         4999575528                       86.14                        20.12
                  measure  no order since June  their mean  their median  other customers  other mean  other median  p-value
      late invoice rate %                   23       45.85         51.85              477       44.94         48.65   0.9299
not fully paid invoices %                   23       32.65         34.21              477       30.49         29.73   0.1740
  outstanding % of billed                   23       22.12         19.96     

In [7]:
import numpy as np

rng = np.random.default_rng(42)
billed_values = cust['billed'].values
outstanding_values = cust['outstanding'].values
n_customers = len(cust)

def unpaid_share(index):
    return outstanding_values[index].sum() / billed_values[index].sum() * 100

for name, mask in [('quiet (ratio above 2)', cust['quiet'].astype(bool)),
                   ('no order since 30 Jun 2024', cust['gone_since_june'].astype(bool))]:
    k = int(mask.sum())
    real = unpaid_share(np.where(mask.values)[0])
    draws = np.array([unpaid_share(rng.choice(n_customers, size=k, replace=False)) for _ in range(5000)])
    print(name, "| customers:", k, "| real:", round(real, 2),
          "| random groups: average", round(draws.mean(), 2),
          ", lowest", round(draws.min(), 2), ", highest", round(draws.max(), 2),
          "| random groups as high as real:", int((draws >= real).sum()), "of 5000")

quiet (ratio above 2) | customers: 67 | real: 21.45 | random groups: average 20.29 , lowest 17.15 , highest 23.29 | random groups as high as real: 414 of 5000
no order since 30 Jun 2024 | customers: 23 | real: 22.53 | random groups: average 20.29 , lowest 15.1 , highest 26.0 | random groups as high as real: 374 of 5000


In [8]:
# keep only invoices dated before 1 July 2024
dates = invoices[['invoice_id', 'invoice_date']].copy()
dates['invoice_date'] = pd.to_datetime(dates['invoice_date'])
early = inv.merge(dates, on='invoice_id', how='left')
early = early[early['invoice_date'] < '2024-07-01']
print('invoices used:', len(early), '| customers:', early['customer_id'].nunique())

# one row per customer, using only those early invoices
by_cust = early.groupby('customer_id').agg(
    invoices=('invoice_id', 'count'),
    with_payment=('has_payment', 'sum'),
    late_invoices=('late', 'sum'),
    not_fully_paid=('not_fully_paid', 'sum'),
    billed=('grand_total', 'sum'),
    paid=('paid_amount', 'sum')
).reset_index()
by_cust['late_rate'] = by_cust['late_invoices'] / by_cust['with_payment'] * 100
by_cust['not_fully_paid_pct'] = by_cust['not_fully_paid'] / by_cust['invoices'] * 100
by_cust['outstanding_pct'] = (by_cust['billed'] - by_cust['paid']) / by_cust['billed'] * 100

# bring in the quiet flag and the no-order-since-30-June flag
flags = silence.copy()
if 'customer_id' not in flags.columns:
    flags = flags.reset_index()
flags['gone_since_june'] = pd.to_datetime(flags['last_order']) < '2024-07-01'
by_cust = by_cust.merge(flags[['customer_id', 'quiet', 'gone_since_june']], on='customer_id', how='left')

# compare each group with the others on the three payment measures
line = 0.05 / 6
print('strict line:', round(line, 4))
for group_col, group_name in [('quiet', 'quiet (67)'), ('gone_since_june', 'no order since 30 Jun 2024 (23)')]:
    print('\n' + group_name)
    for m in ['late_rate', 'not_fully_paid_pct', 'outstanding_pct']:
        a = by_cust.loc[by_cust[group_col] == True, m].dropna()
        b = by_cust.loc[by_cust[group_col] == False, m].dropna()
        p = mannwhitneyu(a, b, alternative='two-sided').pvalue
        print(m, '| group n', len(a), 'mean', round(a.mean(), 2), 'median', round(a.median(), 2),
              '| others n', len(b), 'mean', round(b.mean(), 2), 'median', round(b.median(), 2),
              '| p', round(p, 4))

invoices used: 16106 | customers: 500
strict line: 0.0083

quiet (67)
late_rate | group n 67 mean 45.35 median 50.0 | others n 433 mean 44.69 median 48.39 | p 0.8945
not_fully_paid_pct | group n 67 mean 31.96 median 32.14 | others n 433 mean 30.41 median 30.0 | p 0.2292
outstanding_pct | group n 67 mean 21.52 median 19.9 | others n 433 mean 20.3 median 20.0 | p 0.3588

no order since 30 Jun 2024 (23)
late_rate | group n 23 mean 45.84 median 50.0 | others n 477 mean 44.73 median 48.39 | p 0.8854
not_fully_paid_pct | group n 23 mean 32.59 median 33.33 | others n 477 mean 30.52 median 30.0 | p 0.1879
outstanding_pct | group n 23 mean 22.12 median 20.68 | others n 477 mean 20.38 median 19.94 | p 0.2201


In [9]:
# count blank values in each table we use
for name, table in [('silence', silence), ('inv', inv), ('cust', cust)]:
    print('\n' + name, '| rows:', len(table))
    print(table.isna().sum()[table.isna().sum() > 0])
    print('columns with blanks listed above (nothing listed means no blanks)')

# count zero and negative values in the number columns
for name, table in [('silence', silence), ('inv', inv), ('cust', cust)]:
    nums = table.select_dtypes(include='number')
    print('\n' + name, '| zero and negative values per number column')
    print(pd.DataFrame({'zero': (nums == 0).sum(), 'negative': (nums < 0).sum()}))


silence | rows: 500
Series([], dtype: int64)
columns with blanks listed above (nothing listed means no blanks)

inv | rows: 17640
Series([], dtype: int64)
columns with blanks listed above (nothing listed means no blanks)

cust | rows: 500
Series([], dtype: int64)
columns with blanks listed above (nothing listed means no blanks)

silence | zero and negative values per number column
                  zero  negative
orders               0         0
recency_days         0         0
normal_gap           0         0
silence_ratio        0         0
normal_gap_exact     0         0
ratio_exact          0         0

inv | zero and negative values per number column
             zero  negative
grand_total     0         0
paid_amount  1798         0

cust | zero and negative values per number column
                    zero  negative
invoices               0         0
with_payment           0         0
late_invoices         16         0
not_fully_paid         0         0
billed                 0

In [10]:
# customers with only one order
sil = silence.copy()
if 'customer_id' not in sil.columns:
    sil = sil.reset_index()
print('customers with one order:', (sil['orders'] == 1).sum())
print('fewest orders by any customer:', sil['orders'].min())

# customers sitting on or near the quiet line
near = sil[(sil['ratio_exact'] > 1.99) & (sil['ratio_exact'] < 2.01)][['customer_id', 'orders', 'recency_days', 'ratio_exact']]
print('\ncustomers with ratio exactly 2:', (sil['ratio_exact'] == 2).sum())
print('customers with ratio between 1.99 and 2.01:')
print(near)

# zero paid invoices compared with invoices that have no payment recorded
print('\ninvoices with zero paid amount:', (inv['paid_amount'] == 0).sum())
print('invoices with no payment recorded:', (inv['has_payment'] == False).sum())
print('both at once:', ((inv['paid_amount'] == 0) & (inv['has_payment'] == False)).sum())
print('payment status of the zero paid invoices:')
print(inv.loc[inv['paid_amount'] == 0, 'payment_status'].value_counts())

customers with one order: 0
fewest orders by any customer: 21

customers with ratio exactly 2: 0
customers with ratio between 1.99 and 2.01:
    customer_id  orders  recency_days  ratio_exact
82        C0083      45            89     1.996940
298       C0299      34           124     2.000978
416       C0417      32           112     2.008097

invoices with zero paid amount: 1798
invoices with no payment recorded: 1798
both at once: 1798
payment status of the zero paid invoices:
payment_status
Unpaid    1798
Name: count, dtype: int64


In [11]:
# customer ids should appear once per table
for name, table in [('customers', customers), ('silence', silence), ('cust', cust), ('by_cust', by_cust)]:
    col = table['customer_id'] if 'customer_id' in table.columns else table.index.to_series()
    print(name, '| rows:', len(table), '| repeated customer ids:', col.duplicated().sum())

# invoice ids should appear once per invoice table
for name, table in [('clean_invoices', clean_invoices), ('inv', inv)]:
    print(name, '| rows:', len(table), '| repeated invoice ids:', table['invoice_id'].duplicated().sum())

# every payment row should point to an invoice we kept
outside = (~payments_used['invoice_id'].isin(clean_invoices['invoice_id'])).sum()
print('payment rows pointing to an invoice we did not keep:', outside)

# every invoice in inv should be in clean_invoices, and the reverse
print('invoices in inv but not in clean_invoices:', (~inv['invoice_id'].isin(clean_invoices['invoice_id'])).sum())
print('invoices in clean_invoices but not in inv:', (~clean_invoices['invoice_id'].isin(inv['invoice_id'])).sum())

customers | rows: 500 | repeated customer ids: 0
silence | rows: 500 | repeated customer ids: 0
cust | rows: 500 | repeated customer ids: 0
by_cust | rows: 500 | repeated customer ids: 0
clean_invoices | rows: 17640 | repeated invoice ids: 0
inv | rows: 17640 | repeated invoice ids: 0
payment rows pointing to an invoice we did not keep: 0
invoices in inv but not in clean_invoices: 0
invoices in clean_invoices but not in inv: 0


In [12]:
# customer counts
c = cust.copy()
if 'customer_id' not in c.columns:
    c = c.reset_index()
s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()
c = c.drop(columns=[x for x in ['quiet', 'gone_since_june'] if x in c.columns])
s['gone_since_june'] = pd.to_datetime(s['last_order']) < '2024-07-01'
c = c.merge(s[['customer_id', 'quiet', 'gone_since_june']], on='customer_id', how='left')

print('customers:', len(c))
print('quiet:', c['quiet'].sum(), '| others:', (~c['quiet']).sum(), '| together:', c['quiet'].sum() + (~c['quiet']).sum())
print('no order since 30 Jun 2024:', c['gone_since_june'].sum())
print('of those, inside the quiet group:', (c['gone_since_june'] & c['quiet']).sum())

# money totals compared with the KPI 16 totals
kpi16_billed = 28599637860.60
kpi16_paid = 22795333903.87
kpi16_outstanding = 5804303957

print('\nbilled from invoices:', round(inv['grand_total'].sum(), 2))
print('billed from customer table:', round(c['billed'].sum(), 2))
print('paid from invoices:', round(inv['paid_amount'].sum(), 2))
print('paid from customer table:', round(c['paid'].sum(), 2))
print('billed difference to KPI 16:', round(c['billed'].sum() - kpi16_billed, 2))
print('paid difference to KPI 16:', round(c['paid'].sum() - kpi16_paid, 2))

# outstanding money by group
out_quiet = (c.loc[c['quiet'], 'billed'] - c.loc[c['quiet'], 'paid']).sum()
out_other = (c.loc[~c['quiet'], 'billed'] - c.loc[~c['quiet'], 'paid']).sum()
print('\noutstanding, quiet:', round(out_quiet))
print('outstanding, others:', round(out_other))
print('outstanding, together:', round(out_quiet + out_other))
print('difference to KPI 16 outstanding:', round(out_quiet + out_other - kpi16_outstanding))

customers: 500
quiet: 67 | others: 433 | together: 500
no order since 30 Jun 2024: 23
of those, inside the quiet group: 23

billed from invoices: 28599637860.6
billed from customer table: 28599637860.6
paid from invoices: 22795333903.87
paid from customer table: 22795333903.87
billed difference to KPI 16: 0.0
paid difference to KPI 16: 0.0

outstanding, quiet: 804728429
outstanding, others: 4999575528
outstanding, together: 5804303957
difference to KPI 16 outstanding: 0


In [13]:
from collections import defaultdict
from datetime import date

# silence by plain loop: collect the order dates of every customer
dates_by_customer = defaultdict(list)
for cid, d in zip(orders['customer_id'], pd.to_datetime(orders['order_date'])):
    dates_by_customer[cid].append(d.date())

snap = pd.to_datetime(snapshot).date()
loop_quiet = {}
loop_ratio = {}
for cid, ds in dates_by_customer.items():
    first, last = min(ds), max(ds)
    gap = (last - first).days / (len(ds) - 1)
    recency = (snap - last).days
    loop_ratio[cid] = recency / gap
    loop_quiet[cid] = loop_ratio[cid] > 2

s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()
s['loop_ratio'] = s['customer_id'].map(loop_ratio)
s['loop_quiet'] = s['customer_id'].map(loop_quiet)
print('customers found by the loop:', len(loop_ratio))
print('quiet by the loop:', sum(loop_quiet.values()))
print('customers whose quiet flag differs from the table:', (s['loop_quiet'] != s['quiet']).sum())
print('largest ratio difference:', (s['loop_ratio'] - s['ratio_exact']).abs().max())

# payment measures by plain loop over the invoice rows
billed = defaultdict(float)
paid = defaultdict(float)
count = defaultdict(int)
not_full = defaultdict(int)
for cid, gt, pa, st in zip(inv['customer_id'], inv['grand_total'], inv['paid_amount'], inv['payment_status']):
    billed[cid] += gt
    paid[cid] += pa
    count[cid] += 1
    if st != 'Paid':
        not_full[cid] += 1

c = cust.copy()
if 'customer_id' not in c.columns:
    c = c.reset_index()
c['loop_billed'] = c['customer_id'].map(billed)
c['loop_paid'] = c['customer_id'].map(paid)
c['loop_not_full_pct'] = c['customer_id'].map(lambda k: not_full[k] / count[k] * 100)
c['loop_outstanding_pct'] = c['customer_id'].map(lambda k: (billed[k] - paid[k]) / billed[k] * 100)

print('\nlargest billed difference:', (c['loop_billed'] - c['billed']).abs().max())
print('largest paid difference:', (c['loop_paid'] - c['paid']).abs().max())
print('largest not fully paid % difference:', (c['loop_not_full_pct'] - c['not_fully_paid_pct']).abs().max())
print('largest outstanding % difference:', (c['loop_outstanding_pct'] - c['outstanding_pct']).abs().max())

customers found by the loop: 500
quiet by the loop: 67
customers whose quiet flag differs from the table: 0
largest ratio difference: 0.0

largest billed difference: 4.470348358154297e-08
largest paid difference: 4.470348358154297e-08
largest not fully paid % difference: 0.0
largest outstanding % difference: 4.973799150320701e-14


In [14]:
def customer_measures(invoice_table):
    g = invoice_table.groupby('customer_id')
    out = pd.DataFrame({
        'with_payment': g['has_payment'].apply(lambda x: x.astype(int).sum()),
        'late_invoices': g['late'].apply(lambda x: x.astype(int).sum()),
        'invoices': g['invoice_id'].count(),
        'not_fully_paid': g['not_fully_paid'].apply(lambda x: x.astype(int).sum()),
        'billed': g['grand_total'].sum(),
        'paid': g['paid_amount'].sum()
    })
    out['late_rate'] = out['late_invoices'] / out['with_payment'].replace(0, np.nan) * 100
    out['not_fully_paid_pct'] = out['not_fully_paid'] / out['invoices'] * 100
    out['outstanding_pct'] = (out['billed'] - out['paid']) / out['billed'] * 100
    return out.reset_index()

s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()

# invoice table with each customer's largest invoice removed
largest = inv.groupby('customer_id')['grand_total'].idxmax()
inv_without_largest = inv.drop(index=largest.values)

variants = [
    ('line 2 (normal)', inv, 2.0, None),
    ('line 1.5', inv, 1.5, None),
    ('line 2.5', inv, 2.5, None),
    ('line 2 without C0299', inv, 2.0, 'C0299'),
    ('line 2 without each largest invoice', inv_without_largest, 2.0, None),
]

line = 0.05 / 3
print('strict line:', round(line, 4))
for name, table, cut, drop_id in variants:
    m = customer_measures(table)
    m = m.merge(s[['customer_id', 'ratio_exact']], on='customer_id', how='left')
    if drop_id is not None:
        m = m[m['customer_id'] != drop_id]
    m['quiet_here'] = m['ratio_exact'] > cut
    print('\n' + name, '| quiet:', m['quiet_here'].sum(), '| others:', (~m['quiet_here']).sum())
    for col in ['late_rate', 'not_fully_paid_pct', 'outstanding_pct']:
        a = m.loc[m['quiet_here'], col].dropna()
        b = m.loc[~m['quiet_here'], col].dropna()
        p = mannwhitneyu(a, b, alternative='two-sided').pvalue
        print(col, '| quiet mean', round(a.mean(), 2), '| others mean', round(b.mean(), 2), '| p', round(p, 4))

strict line: 0.0167

line 2 (normal) | quiet: 67 | others: 433
late_rate | quiet mean 45.53 | others mean 44.9 | p 0.929
not_fully_paid_pct | quiet mean 32.01 | others mean 30.37 | p 0.2787
outstanding_pct | quiet mean 21.45 | others mean 20.23 | p 0.3554

line 1.5 | quiet: 119 | others: 381
late_rate | quiet mean 44.5 | others mean 45.14 | p 0.6963
not_fully_paid_pct | quiet mean 31.83 | others mean 30.2 | p 0.0766
outstanding_pct | quiet mean 21.45 | others mean 20.06 | p 0.1012

line 2.5 | quiet: 42 | others: 458
late_rate | quiet mean 46.44 | others mean 44.85 | p 0.6879
not_fully_paid_pct | quiet mean 32.23 | others mean 30.44 | p 0.2023
outstanding_pct | quiet mean 21.49 | others mean 20.29 | p 0.3258

line 2 without C0299 | quiet: 66 | others: 433
late_rate | quiet mean 45.41 | others mean 44.9 | p 0.9441
not_fully_paid_pct | quiet mean 32.08 | others mean 30.37 | p 0.2508
outstanding_pct | quiet mean 21.6 | others mean 20.23 | p 0.2773

line 2 without each largest invoice | qui

In [15]:
# where the data ends
order_dates = pd.to_datetime(orders['order_date'])
inv_dates = pd.to_datetime(invoices['invoice_date'])
pay_date_cols = [c for c in payments.columns if 'date' in c.lower()]
print('payment date columns found:', pay_date_cols)
print('last order date:', order_dates.max().date())
print('last invoice date:', inv_dates.max().date())
for c in pay_date_cols:
    print('last', c + ':', pd.to_datetime(payments[c]).max().date())
print('snapshot used:', snapshot)

# quiet count if the snapshot date moves later
s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()
s['last_order'] = pd.to_datetime(s['last_order'])
for snap_date in ['2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01', '2025-06-01']:
    recency = (pd.Timestamp(snap_date) - s['last_order']).dt.days
    ratio = recency / s['normal_gap_exact']
    print('snapshot', snap_date, '| quiet customers:', (ratio > 2).sum())

# unpaid share by invoice month, last 12 months of data
dated = inv.merge(invoices[['invoice_id', 'invoice_date']], on='invoice_id', how='left')
dated['invoice_date'] = pd.to_datetime(dated['invoice_date'])
dated['month'] = dated['invoice_date'].dt.to_period('M')
by_month = dated.groupby('month').agg(
    invoices=('invoice_id', 'count'),
    unpaid_share=('payment_status', lambda x: round((x == 'Unpaid').mean() * 100, 2)),
    not_fully_paid_share=('not_fully_paid', lambda x: round(x.astype(int).mean() * 100, 2))
)
print('\nunpaid share by invoice month, last 12 months:')
print(by_month.tail(12))

payment date columns found: ['payment_date']
last order date: 2024-12-31
last invoice date: 2025-01-13
last payment_date: 2025-03-20
snapshot used: 2025-01-01 00:00:00
snapshot 2024-12-01 | quiet customers: 42
snapshot 2025-01-01 | quiet customers: 67
snapshot 2025-02-01 | quiet customers: 125
snapshot 2025-03-01 | quiet customers: 201
snapshot 2025-06-01 | quiet customers: 494

unpaid share by invoice month, last 12 months:
         invoices  unpaid_share  not_fully_paid_share
month                                                
2024-02       231         11.69                 32.90
2024-03       254         12.20                 27.17
2024-04       225          9.78                 23.56
2024-05       222         10.36                 29.28
2024-06       226         10.18                 29.20
2024-07       253         10.67                 30.04
2024-08       247          9.72                 35.22
2024-09       242          7.85                 30.99
2024-10       265          7.17

In [16]:
# pick the three customers
s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()
s['last_order'] = pd.to_datetime(s['last_order'])
first_of_23 = s[s['last_order'] < '2024-07-01']['customer_id'].sort_values().iloc[0]
picked = ['C0299', 'C0002', first_of_23]

amount_col = [c for c in payments_used.columns if 'amount' in c.lower()][0]
print('payment amount column used:', amount_col)

for cid in picked:
    print('\n----', cid, '----')

    # raw order dates of this customer
    d = pd.to_datetime(orders.loc[orders['customer_id'] == cid, 'order_date']).sort_values()
    print('delivered orders:', len(d))
    print('first order:', d.iloc[0].date(), '| last order:', d.iloc[-1].date())
    print('days between first and last order:', (d.iloc[-1] - d.iloc[0]).days)
    print('days from last order to 2025-01-01:', (pd.Timestamp('2025-01-01') - d.iloc[-1]).days)

    # table values to compare with
    row = s[s['customer_id'] == cid].iloc[0]
    print('table: orders', row['orders'], '| recency_days', row['recency_days'],
          '| normal_gap_exact', round(row['normal_gap_exact'], 5),
          '| ratio_exact', round(row['ratio_exact'], 5), '| quiet', row['quiet'])

    # raw invoice and payment money of this customer
    mine = clean_invoices[clean_invoices['customer_id'] == cid]
    pays = payments_used[payments_used['invoice_id'].isin(mine['invoice_id'])]
    print('invoices:', len(mine), '| billed:', round(mine['grand_total'].sum(), 2),
          '| paid:', round(pays[amount_col].sum(), 2))

    # table values to compare with
    c = cust.copy()
    if 'customer_id' not in c.columns:
        c = c.reset_index()
    crow = c[c['customer_id'] == cid].iloc[0]
    print('table: invoices', crow['invoices'], '| billed', round(crow['billed'], 2),
          '| paid', round(crow['paid'], 2))

payment amount column used: payment_amount

---- C0299 ----
delivered orders: 34
first order: 2019-01-24 | last order: 2024-08-30
days between first and last order: 2045
days from last order to 2025-01-01: 124
table: orders 34 | recency_days 124 | normal_gap_exact 61.9697 | ratio_exact 2.00098 | quiet True
invoices: 33 | billed: 43979214.8 | paid: 38898802.15
table: invoices 33 | billed 43979214.8 | paid 38898802.15

---- C0002 ----
delivered orders: 51
first order: 2019-01-24 | last order: 2024-09-07
days between first and last order: 2053
days from last order to 2025-01-01: 116
table: orders 51 | recency_days 116 | normal_gap_exact 41.06 | ratio_exact 2.82513 | quiet True
invoices: 49 | billed: 69350700.2 | paid: 52300739.72
table: invoices 49 | billed 69350700.2 | paid 52300739.72

---- C0029 ----
delivered orders: 38
first order: 2019-01-16 | last order: 2024-06-30
days between first and last order: 1992
days from last order to 2025-01-01: 185
table: orders 38 | recency_days 185 | 

In [17]:
due_cols = [c for c in invoices.columns if 'due' in c.lower()]
print('due date columns found:', due_cols)

if len(due_cols) == 0:
    print('no due date column in invoices; columns are:', list(invoices.columns))
else:
    due_col = due_cols[0]
    print('due date column used:', due_col)

    for cid in ['C0299', 'C0002', 'C0029']:
        mine = clean_invoices[clean_invoices['customer_id'] == cid][['invoice_id']]
        mine = mine.merge(invoices[['invoice_id', due_col]], on='invoice_id', how='left')
        pays = payments_used[payments_used['invoice_id'].isin(mine['invoice_id'])].merge(
            mine, on='invoice_id', how='left')
        pays['after_due'] = pd.to_datetime(pays['payment_date']) > pd.to_datetime(pays[due_col])

        with_payment = pays['invoice_id'].nunique()
        late_invoices = pays.loc[pays['after_due'], 'invoice_id'].nunique()

        print('\n----', cid, '----')
        print('raw: invoices with a payment', with_payment,
              '| invoices with a payment after the due date', late_invoices,
              '| late rate', round(late_invoices / with_payment * 100, 2))

        c = cust.copy()
        if 'customer_id' not in c.columns:
            c = c.reset_index()
        row = c[c['customer_id'] == cid].iloc[0]
        print('table: with_payment', row['with_payment'], '| late_invoices', row['late_invoices'],
              '| late_rate', round(row['late_rate'], 2))

due date columns found: ['due_date']
due date column used: due_date

---- C0299 ----
raw: invoices with a payment 32 | invoices with a payment after the due date 17 | late rate 53.12
table: with_payment 32 | late_invoices 17 | late_rate 53.12

---- C0002 ----
raw: invoices with a payment 45 | invoices with a payment after the due date 29 | late rate 64.44
table: with_payment 45 | late_invoices 29 | late_rate 64.44

---- C0029 ----
raw: invoices with a payment 32 | invoices with a payment after the due date 11 | late rate 34.38
table: with_payment 32 | late_invoices 11 | late_rate 34.38


In [18]:
from scipy.stats import fisher_exact

# one row per customer with late rate and the quiet flag
c = cust.copy()
if 'customer_id' not in c.columns:
    c = c.reset_index()
s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()
c = c.drop(columns=[x for x in ['quiet'] if x in c.columns])
c = c.merge(s[['customer_id', 'quiet']], on='customer_id', how='left')

line = 0.05 / 3
print('customers:', len(c), '| quiet:', c['quiet'].sum(), '| quiet share overall:', round(c['quiet'].mean() * 100, 2), '%')
print('strict line:', round(line, 4))

for share in [0.10, 0.25, 0.50]:
    cut = c['late_rate'].quantile(1 - share)
    c['bad_payer'] = c['late_rate'] >= cut
    both = (c['bad_payer'] & c['quiet']).sum()
    bad_only = (c['bad_payer'] & ~c['quiet']).sum()
    quiet_only = (~c['bad_payer'] & c['quiet']).sum()
    neither = (~c['bad_payer'] & ~c['quiet']).sum()
    odds, p = fisher_exact([[both, bad_only], [quiet_only, neither]])
    print('\nbad payers = top', int(share * 100), '% by late rate (late rate at or above', round(cut, 2), ')')
    print('bad payers:', c['bad_payer'].sum(), '| of them quiet:', both,
          '(', round(both / c['bad_payer'].sum() * 100, 2), '% )')
    print('others:', (~c['bad_payer']).sum(), '| of them quiet:', quiet_only,
          '(', round(quiet_only / (~c['bad_payer']).sum() * 100, 2), '% )')
    print('Fisher exact p:', round(p, 4))

customers: 500 | quiet: 67 | quiet share overall: 13.4 %
strict line: 0.0167

bad payers = top 10 % by late rate (late rate at or above 77.8 )
bad payers: 50 | of them quiet: 6 ( 12.0 % )
others: 450 | of them quiet: 61 ( 13.56 % )
Fisher exact p: 1.0

bad payers = top 25 % by late rate (late rate at or above 64.33 )
bad payers: 125 | of them quiet: 17 ( 13.6 % )
others: 375 | of them quiet: 50 ( 13.33 % )
Fisher exact p: 1.0

bad payers = top 50 % by late rate (late rate at or above 49.32 )
bad payers: 250 | of them quiet: 34 ( 13.6 % )
others: 250 | of them quiet: 33 ( 13.2 % )
Fisher exact p: 1.0


In [19]:
s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()
s['last_order'] = pd.to_datetime(s['last_order'])

print('earliest last order of any customer:', s['last_order'].min().date())
print('latest last order of any customer:', s['last_order'].max().date())
print('customers whose last order is in 2024:', (s['last_order'] >= '2024-01-01').sum(), 'of', len(s))
print('customers whose last order is before 2024:', (s['last_order'] < '2024-01-01').sum())

print('\ncustomers by days since last order (snapshot 2025-01-01):')
for days in [30, 90, 120, 180]:
    n = (s['recency_days'] <= days).sum()
    print('ordered within', days, 'days:', n, '(', round(n / len(s) * 100, 2), '% )')
print('longest silence of any customer:', s['recency_days'].max(), 'days')
print('customers silent for more than 180 days:', (s['recency_days'] > 180).sum())

earliest last order of any customer: 2024-01-08
latest last order of any customer: 2024-12-31
customers whose last order is in 2024: 500 of 500
customers whose last order is before 2024: 0

customers by days since last order (snapshot 2025-01-01):
ordered within 30 days: 167 ( 33.4 % )
ordered within 90 days: 385 ( 77.0 % )
ordered within 120 days: 433 ( 86.6 % )
ordered within 180 days: 475 ( 95.0 % )
longest silence of any customer: 359 days
customers silent for more than 180 days: 25


In [20]:
from scipy.stats import fisher_exact

# invoices with their invoice date
dated = inv.merge(invoices[['invoice_id', 'invoice_date']], on='invoice_id', how='left')
dated['invoice_date'] = pd.to_datetime(dated['invoice_date'])
dated['recent'] = dated['invoice_date'] >= '2024-11-14'
print('recent invoices (from 2024-11-14):', dated['recent'].sum(), '| earlier invoices:', (~dated['recent']).sum())

line = 0.05 / 2
print('strict line:', round(line, 3))

# unpaid share: recent against earlier
r = dated[dated['recent']]
e = dated[~dated['recent']]
unpaid_r = (r['payment_status'] == 'Unpaid').sum()
unpaid_e = (e['payment_status'] == 'Unpaid').sum()
odds, p = fisher_exact([[unpaid_r, len(r) - unpaid_r], [unpaid_e, len(e) - unpaid_e]])
print('\nunpaid share | recent', round(unpaid_r / len(r) * 100, 2), '% | earlier', round(unpaid_e / len(e) * 100, 2), '% | p', round(p, 4))

# late share among invoices that have a payment: recent against earlier
rp = r[r['has_payment'] == True]
ep = e[e['has_payment'] == True]
late_r = rp['late'].astype(bool).sum()
late_e = ep['late'].astype(bool).sum()
odds, p = fisher_exact([[late_r, len(rp) - late_r], [late_e, len(ep) - late_e]])
print('late share among paid invoices | recent', round(late_r / len(rp) * 100, 2), '% | earlier', round(late_e / len(ep) * 100, 2), '% | p', round(p, 4))

# days from invoice date to each payment
pays = payments_used.merge(invoices[['invoice_id', 'invoice_date']], on='invoice_id', how='left')
pays['days_to_pay'] = (pd.to_datetime(pays['payment_date']) - pd.to_datetime(pays['invoice_date'])).dt.days
print('\ndays from invoice to payment | payments:', len(pays))
print('median:', pays['days_to_pay'].median(), '| 95th percentile:', pays['days_to_pay'].quantile(0.95),
      '| 99th percentile:', pays['days_to_pay'].quantile(0.99), '| longest:', pays['days_to_pay'].max())
print('payments that came more than 66 days after the invoice:', (pays['days_to_pay'] > 66).sum(),
      '(', round((pays['days_to_pay'] > 66).mean() * 100, 2), '% )')

recent invoices (from 2024-11-14): 427 | earlier invoices: 17213
strict line: 0.025

unpaid share | recent 9.37 % | earlier 10.21 % | p 0.6272
late share among paid invoices | recent 45.99 % | earlier 45.31 % | p 0.7962

days from invoice to payment | payments: 18840
median: 32.0 | 95th percentile: 66.0 | 99th percentile: 85.0 | longest: 90
payments that came more than 66 days after the invoice: 918 ( 4.87 % )


In [21]:
# keep only invoices that had at least 90 days to be paid
kept = dated[dated['invoice_date'] <= '2024-12-20']
removed = len(dated) - len(kept)
print('invoices removed (dated after 2024-12-20):', removed, '| share of all invoices:', round(removed / len(dated) * 100, 2), '%')
print('invoices kept:', len(kept))

m = customer_measures(kept)
s = silence.copy()
if 'customer_id' not in s.columns:
    s = s.reset_index()
m = m.merge(s[['customer_id', 'ratio_exact']], on='customer_id', how='left')
m['quiet_here'] = m['ratio_exact'] > 2

line = 0.05 / 3
print('\ncustomers:', len(m), '| quiet:', m['quiet_here'].sum(), '| others:', (~m['quiet_here']).sum())
print('strict line:', round(line, 4))
for col in ['late_rate', 'not_fully_paid_pct', 'outstanding_pct']:
    a = m.loc[m['quiet_here'], col].dropna()
    b = m.loc[~m['quiet_here'], col].dropna()
    p = mannwhitneyu(a, b, alternative='two-sided').pvalue
    print(col, '| quiet mean', round(a.mean(), 2), '| others mean', round(b.mean(), 2), '| p', round(p, 4))


invoices removed (dated after 2024-12-20): 136 | share of all invoices: 0.77 %
invoices kept: 17504

customers: 500 | quiet: 67 | others: 433
strict line: 0.0167
late_rate | quiet mean 45.53 | others mean 44.94 | p 0.9359
not_fully_paid_pct | quiet mean 32.01 | others mean 30.38 | p 0.2836
outstanding_pct | quiet mean 21.45 | others mean 20.26 | p 0.3578


In [22]:
# due period of each invoice, in days between invoice date and due date
due = invoices[['invoice_id', 'due_date']].copy()
due['due_date'] = pd.to_datetime(due['due_date'])
t = dated.merge(due, on='invoice_id', how='left')
t['due_period_days'] = (t['due_date'] - t['invoice_date']).dt.days
print('invoices per due period (days):')
print(t['due_period_days'].value_counts().sort_index())

# late share among invoices that have a payment, by due period
paid_t = t[t['has_payment'] == True].copy()
paid_t['late_flag'] = paid_t['late'].astype(bool)
counts = paid_t.groupby('due_period_days')['late_flag'].agg(['count', 'sum'])
counts.columns = ['invoices_with_payment', 'late_invoices']
counts['late_share_pct'] = (counts['late_invoices'] / counts['invoices_with_payment'] * 100).round(2)
print('\nlate share among invoices with a payment, by due period:')
print(counts)

# chi-square test across the due periods
table = np.array([counts['late_invoices'], counts['invoices_with_payment'] - counts['late_invoices']])
chi2, p, dof, expected = chi2_contingency(table)
print('\nchi-square:', round(chi2, 3), '| degrees of freedom:', dof, '| p:', p)
print('smallest expected count:', round(expected.min(), 2))

invoices per due period (days):
due_period_days
15    3094
30    6858
45    3967
60    3721
Name: count, dtype: int64

late share among invoices with a payment, by due period:
                 invoices_with_payment  late_invoices  late_share_pct
due_period_days                                                      
15                                2772           2178           78.57
30                                6195           3556           57.40
45                                3549           1208           34.04
60                                3326            238            7.16

chi-square: 3738.787 | degrees of freedom: 3 | p: 0.0
smallest expected count: 1256.34


In [23]:
# due period of each invoice, in days between invoice date and due date
due = invoices[['invoice_id', 'due_date']].copy()
due['due_date'] = pd.to_datetime(due['due_date'])
t = dated.merge(due, on='invoice_id', how='left')
t['due_period_days'] = (t['due_date'] - t['invoice_date']).dt.days
print('invoices per due period (days):')
print(t['due_period_days'].value_counts().sort_index())

# late share among invoices that have a payment, by due period
paid_t = t[t['has_payment'] == True].copy()
paid_t['late_flag'] = paid_t['late'].astype(bool)
counts = paid_t.groupby('due_period_days')['late_flag'].agg(['count', 'sum'])
counts.columns = ['invoices_with_payment', 'late_invoices']
counts['late_share_pct'] = (counts['late_invoices'] / counts['invoices_with_payment'] * 100).round(2)
print('\nlate share among invoices with a payment, by due period:')
print(counts)

# chi-square test across the due periods
table = np.array([counts['late_invoices'], counts['invoices_with_payment'] - counts['late_invoices']])
chi2, p, dof, expected = chi2_contingency(table)
print('\nchi-square:', round(chi2, 3), '| degrees of freedom:', dof, '| p:', p)
print('smallest expected count:', round(expected.min(), 2))

invoices per due period (days):
due_period_days
15    3094
30    6858
45    3967
60    3721
Name: count, dtype: int64

late share among invoices with a payment, by due period:
                 invoices_with_payment  late_invoices  late_share_pct
due_period_days                                                      
15                                2772           2178           78.57
30                                6195           3556           57.40
45                                3549           1208           34.04
60                                3326            238            7.16

chi-square: 3738.787 | degrees of freedom: 3 | p: 0.0
smallest expected count: 1256.34


In [24]:
print('columns in the customers table:')
print(list(customers.columns))
print()
print(customers.dtypes)

# any column whose name suggests that a customer is active, inactive or has left
words = ['status', 'active', 'inactive', 'churn', 'left', 'closed', 'end', 'terminated', 'stop', 'flag']
hits = [c for c in customers.columns if any(w in c.lower() for w in words)]
print('\ncolumns that look like a status or leaving field:', hits)
for c in hits:
    print('\nvalues in', c)
    print(customers[c].value_counts(dropna=False))

columns in the customers table:
['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']

customer_id             object
customer_type           object
industry_segment        object
city                    object
state                   object
pincode                  int64
region                  object
branch_id               object
credit_limit             int64
current_balance          int64
payment_terms           object
customer_since          object
last_purchase_date      object
total_purchase_value     int64
customer_rating          int64
dtype: object

columns that look like a status or leaving field: []


In [25]:
# columns of the orders, invoices and payments tables, with the same name search
words = ['status', 'active', 'inactive', 'churn', 'left', 'closed', 'end', 'terminated', 'stop', 'flag']
for name, table in [('so_h (orders)', so_h), ('invoices', invoices), ('payments', payments)]:
    print('\n' + name, '| columns:', list(table.columns))
    hits = [c for c in table.columns if any(w in c.lower() for w in words)]
    print('columns that look like a status or leaving field:', hits)
    for c in hits:
        print('values in', c)
        print(table[c].value_counts(dropna=False))


so_h (orders) | columns: ['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']
columns that look like a status or leaving field: ['order_status']
values in order_status
order_status
Delivered    18033
Cancelled     1967
Name: count, dtype: int64

invoices | columns: ['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']
columns that look like a status or leaving field: ['payment_status']
values in payment_status
payment_status
Paid              12536
Partially Paid     3660
Unpaid             1837
Name: count, dtype: int64

payments | columns: ['payment_id', 'invoice_id', 'payment_date', 'payment_amount', 'payment_method']
columns that look like a status or leaving field: []
